# 第5章 利率期限结构与曲线构建 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch05_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch05_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：bootstrap 复现例5.1 + 定价自检


In [ ]:
import numpy as np
from fi import curve, plotting
plotting.use_chinese_style()
par = [0.02, 0.025, 0.028]
zeros, dfs = curve.bootstrap(par)
print('即期利率:', [f'{z*100:.4f}%' for z in zeros])
for n, c in enumerate(par, start=1):
    pv = c*dfs[:n-1].sum() + (1+c)*dfs[n-1]
    print(f'{n}yr 平价债重新定价 = {pv:.8f}  (应=1)')


## 编程实验 8：线性 vs 三次样条插值的远期光滑性


In [ ]:
from fi import data
cv = data.load_sample('cgb_yield_curve')
ten = np.arange(1, 11)
par_int = curve.interpolate(cv['tenor'], cv['yield_pct']/100, ten, 'linear')
z_int, _ = curve.bootstrap(par_int)
fine = np.linspace(1, 10, 91)
z_lin = curve.interpolate(ten, z_int, fine, 'linear'); z_cub = curve.interpolate(ten, z_int, fine, 'cubic')
def fwd(t, z):
    df = (1+z)**(-t); return (df[:-1]/df[1:])**(1/(t[1:]-t[:-1])) - 1
fig, _ = plotting.new_axes(figsize=(9,4)); fig.clf()
a1 = fig.add_subplot(1,2,1); a1.plot(fine, z_lin*100, label='线性'); a1.plot(fine, z_cub*100, label='样条'); a1.set_title('即期插值'); a1.legend()
a2 = fig.add_subplot(1,2,2); a2.plot(fine[1:], fwd(fine,z_lin)*100, label='线性→远期'); a2.plot(fine[1:], fwd(fine,z_cub)*100, label='样条→远期'); a2.set_title('远期光滑性'); a2.legend()
fig.tight_layout()


## 编程实验 9：par/spot/forward 三条曲线


In [ ]:
par_c = curve.interpolate(cv['tenor'], cv['yield_pct']/100, ten, 'linear')
z_c, _ = curve.bootstrap(par_c); ft, fwdc = curve.forward_curve(z_c)
fig, ax = plotting.new_axes()
ax.plot(ten, par_c*100, marker='o', label='到期收益率(平价)')
ax.plot(ten, z_c*100, marker='^', label='即期')
ax.plot(ft, fwdc*100, marker='s', ls='--', label='远期')
ax.set_xlabel('期限（年）'); ax.set_ylabel('利率 (%)'); ax.set_title('三条曲线 par<spot<forward'); ax.legend()
fig.tight_layout()
print('akshare 真实国债收益率可替换 cv，重做并讨论当前曲线形态隐含的市场预期')
